# Phase 3: commitment tracking

Release `data/results/2026-09-23/`. Plan: `plans/phase_3.md` with its amendment. Computation: `analysis/phase3_extract.py`, `analysis/phase3_track.py`; outputs in `data/phase3/`. This phase is descriptive: there is no test and nothing enters the register.

The main deliverable is `data/phase3/threads.csv`, one row per commitment thread with rule-based proposed statuses and empty columns for review. **Every status in this notebook is a proposal of the rules, not a finding**, until the table has been reviewed.

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sys

ROOT = Path.cwd().parents[1] if Path.cwd().name == "roman" else Path.cwd()
sys.path.insert(0, str(ROOT / "notebooks" / "roman" / "analysis"))
import phase3_extract as p3

OUT = ROOT / "notebooks" / "roman" / "data" / "phase3"
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.precision", 2)
st = pd.read_csv(OUT / "statements.csv")
cand = pd.read_csv(OUT / "candidates.csv")
threads = pd.read_csv(OUT / "threads.csv")
fups = pd.read_csv(OUT / "followups.csv")

## Extraction rules

In [ ]:
pd.Series({"(a) quantity": p3.QTY_RE.pattern, "(b) year": p3.YEAR_RE.pattern, "(b) quarter": p3.QUARTER_RE.pattern,
           **{f"(b) {k}": v.pattern for k, v in p3.RELATIVE.items()}, "(c) forward-looking": p3.FORWARD_RE.pattern},
          name="regular expression").to_frame()

In [ ]:
pd.DataFrame(p3.METRICS, columns=["metric group", "type", "pattern"])

A candidate needs all three parts in the same sentence. Sentences split at `ex.` are joined with the next sentence of the same statement (32 joins), sentences ending in `No.` are not; the JPMorgan event call contributes candidates as origins only.

## Candidates

In [ ]:
n_mgmt = st[st.call_type == "earnings"].groupby("bank").size()
tab = cand.groupby(["bank", "call_type", "type"]).size().unstack("type", fill_value=0)
tab["per 1,000 management sentences"] = tab.sum(axis=1) / tab.index.get_level_values("bank").map(n_mgmt) * 1000
tab

In [ ]:
cand.groupby(["quarter", "bank"]).size().unstack("bank", fill_value=0)

UBS management makes 106 candidate statements against 27 at JPMorgan in the earnings calls, 30 per 1,000 management sentences against 9, and most UBS candidates are targets while most JPMorgan candidates are guidance. The UBS count peaks in the 2023-Q4 call (30), when the multi-year integration plan was presented, and in 2024-Q4 (25).

## Threads and proposed statuses

In [ ]:
pd.read_csv(OUT / "summary_threads.csv")

In [ ]:
pd.read_csv(OUT / "summary_status.csv").pivot_table(index="proposed_final_status", columns=["bank", "type"],
                                                    values="threads", fill_value=0)

The rules group the UBS targets into 36 threads and the JPMorgan guidance into 15. JPMorgan guidance is followed up in 1.7 later calls on average and revised in most threads, which is the ordinary quarterly guidance cycle; UBS targets are followed up in 2.3 later calls.

In [ ]:
sens = pd.read_csv(OUT / "sensitivity_thresholds.csv")
sens["no_follow_up"] = sens.proposed_final_status.str.startswith("no_follow_up")
pd.concat({"threads": sens.groupby(["threshold", "bank", "type"]).threads.sum().unstack(["bank", "type"]),
           "no follow-up found": sens[sens.no_follow_up].groupby(["threshold", "bank", "type"]).threads.sum()
           .unstack(["bank", "type"])}, axis=1).fillna(0).astype(int)

The counts depend strongly on the similarity threshold: UBS target threads range from 17 to 56 and those without any follow-up from 2 to 36 between thresholds 0.5 and 0.7. Counts of threads and of “no follow-up” are therefore not robust and are not interpreted as such; the review of `threads.csv` decides.

In [ ]:
status_order = ["restated", "revised_up", "revised_down", "horizon_moved", "achieved", "mentioned_no_value"]
colors = dict(zip(status_order, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#b0b0b0"]))
quarters = sorted(st[st.call_type == "earnings"].quarter.unique())
fig, axes = plt.subplots(1, 2, figsize=(13, 9), gridspec_kw={"width_ratios": [1, 1]})
for ax, bank in zip(axes, ["UBS", "JPM"]):
    t = threads[(threads.bank == bank) & threads.type.isin(["target", "guidance"])].sort_values(
        ["type", "first_call_date", "thread"], ascending=[False, True, True]).reset_index(drop=True)
    for y, r in t.iterrows():
        x0 = quarters.index(r.origin_call.split()[0])
        ax.scatter(x0, y, marker="D" if r.origin_only else "o", facecolor="white", edgecolor="0.2", s=40, zorder=3)
        for _, f in fups[fups.thread == r.thread].iterrows():
            ax.scatter(quarters.index(f.quarter), y, color=colors[f.proposed_status], s=40, zorder=3)
        ax.plot([x0, len(quarters) - 1], [y, y], color="0.92", lw=1, zorder=1)
    ax.set_yticks(range(len(t)))
    ax.set_yticklabels([f"{r.metric} {r.value:g} {r.unit}" for _, r in t.iterrows()], fontsize=7)
    ax.set_xticks(range(len(quarters))); ax.set_xticklabels(quarters, rotation=45, fontsize=8)
    ax.invert_yaxis(); ax.set_title(f"{bank}: targets, then guidance", loc="left", fontsize=10)
    ax.spines[["top", "right"]].set_visible(False)
handles = [plt.Line2D([], [], marker="o", ls="", color=colors[s], label=s) for s in status_order]
handles += [plt.Line2D([], [], marker="o", ls="", markerfacecolor="white", markeredgecolor="0.2", label="first mention"),
            plt.Line2D([], [], marker="D", ls="", markerfacecolor="white", markeredgecolor="0.2",
                       label="first mention in event call")]
fig.legend(handles=handles, loc="lower center", ncol=4, frameon=False, fontsize=9)
fig.supxlabel("Earnings call (reporting quarter)", y=0.07, fontsize=10)
plt.tight_layout(rect=(0, 0.08, 1, 1))

Each row is a thread, labelled by metric group and the first value; open marks are first mentions, filled marks the proposed status of the follow-up in a later call. Reading the rows against the texts shows frequent rule errors (a CET1 sentence filed under cost saves because it mentions cost saves, reported results matched as revisions of targets), so the chart shows where to look, not what happened.

## Case studies (read in full)

**UBS gross cost saves.** In 2023-Q2 Ermotti set the aim to “take out over 10 billion dollars in gross expenses” against the 2022 base; in 2023-Q4 this became “around 13 billion in gross cost reductions by the end of 2026”. Progress was then reported against the 13 billion in every call: 5 billion or nearly 40 % (2024-Q1), 6 billion or around 45 % (2024-Q2), past the halfway mark (2024-Q3) and almost 60 % (2024-Q4), a consistently tracked and once raised target.

**UBS integration-related expenses.** The cost of achieving the saves was put at around 13 billion by the end of 2026 in 2023-Q4 and restated in 2024-Q1 as a 1-to-1 ratio of costs to achieve and gross saves. In 2024-Q4 management said it “now expect[s] cumulative integration-related expenses to total around 14 billion”, a 1 billion increase in the cost of the integration with an unchanged savings target.

**UBS Basel III final (FRTB).** The expected day-1 increase in risk-weighted assets was “roughly 5 %” in 2023-Q3, “around 15 billion” in the core businesses in 2023-Q4 and “around 5 % of RWA” or 25 billion in 2024-Q2; in 2024-Q4 the bank said it intends to report “around 1 billion of incremental RWA, broadly neutral to our CET1 capital ratio”, with FRTB adding 9 billion and operational risk falling by 7 billion. The regulatory impact on capital was thus revised down by an order of magnitude within one year, which the phase 1 capital-regulation lens did not show.

**UBS AT1 issuance, a miss of the rules.** In 2023-Q4 UBS expected “to issue up to 2 billion in AT1 in 2024”; the rules find no follow-up, yet the 2024-Q1 call reports 1.5 billion placed in February and the 2024-Q4 call 3.5 billion issued during the year. The target was exceeded, and the case shows that “no follow-up found” must not be read as “dropped” without review.

**JPMorgan: guidance cycle and First Republic.** JPMorgan revised its 2023 NII ex. Markets guidance upwards in each 2023 call (about 81, 87 and 89 billion) and moved its 2024 adjusted expense guidance between about 90 and 92 billion (90, 91, 92, 91.5), which is ordinary forecast updating. The merger-specific statements of the event call (restructuring costs of about 2 billion post-tax over 2023 and 2024, more than 500 million of incremental net income per year) are never restated as numbers in the earnings calls; instead, First Republic's quarterly revenue, expense and net income are reported until 2024-Q1 and then no longer separately.

## Validation

`data/labelling/commitments.csv` holds 203 management sentences, blind and in random order: 104 candidates stratified by bank and type, and per bank about 25 non-candidates with a quantity and a future horizon and 25 with a quantity only. The key with strata and weights is `commitments_key.csv`, the guide `commitments_guide.md`. The sample will not be labelled (plan amendment of 2026-09-24), so extraction precision and recall are not validated; the review of `threads.csv` gives the agreement between proposed and reviewed statuses.